# topk-predictions — worked example 2: Ranked predictions as a label list

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `topk-predictions`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`topk` returns indices already sorted from highest to lowest score (when `sorted=True`, the default). So `logits.topk(k, dim=-1).indices` is directly the ranked list of predicted class ids — most confident first. This is exactly the format a retrieval or recommendation system emits.

## Worked solution

We turn a single row of logits into its ranked top-k label list.

1. `logits.topk(k, dim=-1)` returns values and indices sorted descending by score, so `.indices` is the ranking we want.
2. We verify the ranking is genuine: gathering the logits at those indices must come back in non-increasing order, which we check against `.values`.
3. We print the ranked label ids and the matching scores side by side.

The printed indices are the model's class guesses from most to least confident.

In [ ]:
import torch as t

t.manual_seed(1)

def ranked_labels(logits: t.Tensor, k: int) -> t.Tensor:
    return logits.topk(k, dim=-1).indices

logits = t.tensor([0.1, 3.2, 1.5, -0.4, 2.7])
ranked = ranked_labels(logits, 3)
print('ranked label ids:', ranked.tolist())
print('their scores:', logits[ranked].tolist())
print('sorted desc:', bool((logits[ranked][:-1] >= logits[ranked][1:]).all()))